# Grover search

Amplify the marked two-qubit state and compare the complete ideal distribution.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

Grover's iterate combines a phase oracle with inversion about the mean to amplify one marked basis state.

In [2]:
circuit = QuantumCircuit(2)
circuit.h(range(2))
circuit.cz(0, 1)  # mark |11>
circuit.h(range(2))
circuit.x(range(2))
circuit.cz(0, 1)
circuit.x(range(2))
circuit.h(range(2))

def get_reference():
    return Statevector.from_instruction(circuit).probabilities()

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(get_reference)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def get_mettleq():
    state = backend.run(compiled, shots=1, return_statevector=True).result().data(0)["statevector"]
    return np.abs(np.asarray(state)) ** 2

candidate, mettleq_ms, _ = benchmark(get_mettleq)
error = max_abs_error(reference, candidate)
marked = format(int(np.argmax(candidate)), "02b")
method, device = qiskit_selection(backend)

## 4. Check correctness before discussing speed

The state probabilities must agree and both simulators must identify the same marked item.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/07_grover_search.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="probability vector atol=2e-6 and exact marked item",
    passed=error <= 2e-6 and marked == "11",
    exact_match=marked == "11",
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "marked_item": marked, "marked_probability": candidate[-1]},
)


Comparison summary
------------------
Correctness contract: PASS — probability vector atol=2e-6 and exact marked item
SDK reference median: 0.190 ms
MettleQ median:       0.502 ms
Timing interpretation: the SDK reference was 2.650x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: yes

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "probability vector atol=2e-6 and exact marked item", "exact_match": true, "framework": "qiskit", "machine": "arm64", "metrics": {"marked_item": "11", "marked_probability": 0.9999997615814209, "max_probability_error": 2.3841857821338408e-07}, "mettleq_median_ms": 0.502375012729317, "notebook": "qiskit/07_grover_search.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 0.18958401051349938, "reference_over_mettleq": 0.3773754778995119, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

Larger Grover simulations can benefit from statevector acceleration; this two-qubit example is deliberately inspectable.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.